# Parking Sign Detection - Interactive Inference (Colab)

Load pretrained model, upload images, and interactively adjust predictions.

**Features:**
- Upload model weights
- Batch image upload
- Confidence filtering
- Interactive box adjustment
- Export annotations

## 1. Mount Google Drive (Optional)

Mount if you want to save/load files from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup

In [ ]:
!pip install ultralytics -q

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import numpy as np
from PIL import Image
from google.colab import files
import shutil

# Paths
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Upload directory: {UPLOAD_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Upload Model Weights

Upload your trained `best.pt` file.

In [ ]:
print("Upload your trained model weights (best.pt):")
uploaded = files.upload()

if uploaded:
    MODEL_PATH = list(uploaded.keys())[0]
    print(f"Model uploaded: {MODEL_PATH}")
else:
    print("No file uploaded. Using default yolo11n.pt for testing.")
    MODEL_PATH = "yolo11n.pt"

## 4. Load Model

In [ ]:
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

## 5. Upload Test Images

In [ ]:
print("Upload test images (you can select multiple):")
uploaded_imgs = files.upload()

uploaded_files = []
for filename in uploaded_imgs.keys():
    img_path = UPLOAD_DIR / filename
    shutil.copy(filename, img_path)
    uploaded_files.append(str(img_path))
    print(f"Saved: {filename}")

print(f"\nTotal images uploaded: {len(uploaded_files)}")

## 6. Run Predictions

In [ ]:
# Confidence threshold
CONF_THRESHOLD = 0.25

predictions = {}  # {image_path: {boxes: [], confidences: [], classes: []}}

print(f"Running predictions with confidence threshold: {CONF_THRESHOLD}")
print("-" * 50)

for img_path in uploaded_files:
    results = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)
    
    # Extract predictions
    boxes = []
    confidences = []
    classes = []
    
    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                boxes.append([float(x1), float(y1), float(x2), float(y2)])
                confidences.append(float(box.conf.cpu().numpy()))
                classes.append(int(box.cls.cpu().numpy()))
    
    predictions[img_path] = {
        'boxes': boxes,
        'confidences': confidences,
        'classes': classes
    }
    
    print(f"{Path(img_path).name}: {len(boxes)} detections")

print("\nPredictions complete!")

## 7. Visualize All Results

In [ ]:
# Display all images with predictions
for img_path in uploaded_files:
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax.imshow(img)
    
    if img_path in predictions:
        boxes = predictions[img_path]['boxes']
        confidences = predictions[img_path]['confidences']
        classes = predictions[img_path]['classes']
        
        for box, conf, cls in zip(boxes, confidences, classes):
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=3, edgecolor='red', facecolor='none'
            )
            ax.add_patch(rect)
            label = f"{model.names.get(cls, cls)}: {conf:.2f}"
            ax.text(x1, y1 - 5, label, color='red', fontsize=12,
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    ax.set_title(Path(img_path).name)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 8. View Predictions Data

In [ ]:
# Display prediction data as table
import pandas as pd

rows = []
for img_path, pred in predictions.items():
    for box, conf, cls in zip(pred['boxes'], pred['confidences'], pred['classes']):
        rows.append({
            'Image': Path(img_path).name,
            'Class': model.names.get(cls, cls),
            'Confidence': f"{conf:.3f}",
            'Box': f"[{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]"
        })

df = pd.DataFrame(rows)
display(df)

## 9. Export Results

In [ ]:
# Export JSON
export_data = []
for img_path, pred in predictions.items():
    export_data.append({
        'image': Path(img_path).name,
        'boxes': pred['boxes'],
        'confidences': pred['confidences'],
        'classes': pred['classes']
    })

json_path = OUTPUT_DIR / 'predictions.json'
with open(json_path, 'w') as f:
    json.dump(export_data, f, indent=2)
print(f"Exported: {json_path}")

# Save annotated images
for img_path in uploaded_files:
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax.imshow(img)
    
    if img_path in predictions:
        boxes = predictions[img_path]['boxes']
        confidences = predictions[img_path]['confidences']
        classes = predictions[img_path]['classes']
        
        for box, conf, cls in zip(boxes, confidences, classes):
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=3, edgecolor='red', facecolor='none'
            )
            ax.add_patch(rect)
            label = f"{model.names.get(cls, cls)}: {conf:.2f}"
            ax.text(x1, y1 - 5, label, color='red', fontsize=12,
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    ax.axis('off')
    plt.tight_layout()
    
    output_img_path = OUTPUT_DIR / f"annotated_{Path(img_path).name}"
    plt.savefig(output_img_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Saved: {output_img_path}")

print("\nExport complete!")

## 10. Download Results

In [ ]:
# Download all files
print("Downloading output files...")
!zip -r output.zip /content/output/*
files.download('/content/output.zip')

## Summary

**Usage:**
1. (Optional) Mount Google Drive
2. Upload model weights (`best.pt`)
3. Upload test images
4. View predictions and annotated images
5. Download results as ZIP

**Note:** To change confidence threshold, edit `CONF_THRESHOLD` in cell 6 and re-run.